# LLM Fundamentals + Resume RAG Pipeline on Databricks

**Part 1 – LLM basics:** prompts, tokens, context window, temperature, a first LLM call.
**Part 2 – RAG pipeline:** Bronze (raw PDFs) → Silver (extracted text) → Gold (chunks) → embeddings → vector search → LLM answer.

> Put your resume PDFs in the volume created in section 2.1 before running the Bronze step.
> Outputs are cleared in this file because the original contained candidate PII.

## 0. Setup

In [ ]:
%pip install pypdf langchain-text-splitters langchain-core databricks-sdk numpy

In [ ]:
%restart_python

### 0.1 Configuration and shared helpers

In [ ]:
import time
import requests
import numpy as np
from databricks.sdk import WorkspaceClient

# ---- Configuration -------------------------------------------------
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "rag_demo"
VOLUME = "raw_docs"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_documents"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_documents"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_resume_chunks"
EMBEDDED_TABLE = f"{CATALOG}.{SCHEMA}.gold_resume_chunks_embedded"

CHAT_MODEL = "system.ai.gpt-oss-120b"
EMBEDDING_MODEL = "system.ai.gte-large-en"

w = WorkspaceClient()
BASE_URL = w.config.host.rstrip("/")


def _post(path, payload, timeout=60):
    """POST to the Databricks AI Gateway using notebook authentication."""
    return requests.post(
        f"{BASE_URL}/ai-gateway/mlflow/v1/{path}",
        headers={**w.config.authenticate(), "Content-Type": "application/json"},
        json=payload,
        timeout=timeout,
    )


def chat(messages, temperature=0, max_tokens=None, timeout=120):
    """Call the hosted chat model and return the answer text."""
    payload = {"model": CHAT_MODEL, "messages": messages, "temperature": temperature}
    if max_tokens:
        payload["max_tokens"] = max_tokens
    response = _post("chat/completions", payload, timeout=timeout)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


def embed_texts(texts, batch_size=8, max_retries=5, verbose=False):
    """
    Embed a list of texts. Small batches + retry on HTTP 429 because larger
    batches exceed the workspace QPS limit (REQUEST_LIMIT_EXCEEDED).
    Returns vectors in the same order as `texts`.
    """
    texts = list(texts)
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        for attempt in range(max_retries):
            response = _post("embeddings", {"model": EMBEDDING_MODEL, "input": batch})
            if response.status_code != 429:
                break
            wait = float(response.headers.get("Retry-After", min(15 * 2 ** attempt, 120)))
            print(f"Rate limited (429) - retrying in {wait:.0f}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)

        response.raise_for_status()
        data = sorted(response.json()["data"], key=lambda item: item["index"])
        vectors.extend([float(x) for x in item["embedding"]] for item in data)

        if verbose:
            print(f"Embedded {min(start + batch_size, len(texts))} / {len(texts)}")
    return vectors


print("Catalog:", CATALOG, "| Schema:", SCHEMA)

## Part 1 - LLM fundamentals

### 1.1 Prompts

In [ ]:
prompt = """
You are a healthcare data assistant.

Explain what a healthcare claim is
in simple language.
"""

print(prompt)

### 1.2 System instruction vs. user question

In [ ]:
user_question = "What is a healthcare claim?"

system_instruction = """
Answer in simple language.
Use examples where possible.
"""

print("System instruction:")
print(system_instruction)

print("User question:")
print(user_question)

### 1.3 Tokens
Whitespace splitting is only an approximation. Real models use sub-word tokenizers,
so the true token count is usually higher than the word count.

In [ ]:
text = "I am learning RAG using Databricks"

# Simplified whitespace-based splitting
tokens = text.split()

print("Original text:", text)
print("Tokens:", tokens)
print("Approximate token count:", len(tokens))

### 1.4 Context window budget
Input (system + question + document) and the model's response must all fit in the context window.

In [ ]:
context_window = 1000

system_tokens = 100
question_tokens = 100
document_tokens = 500
response_tokens = 300

input_tokens = system_tokens + question_tokens + document_tokens
total_tokens = input_tokens + response_tokens

# Room left for the response after the input is counted
max_response_tokens = context_window - input_tokens

print("Context window:", context_window)
print("Input tokens:", input_tokens)
print("Max response tokens available:", max_response_tokens)
print("Total required tokens:", total_tokens)
print("Remaining tokens:", context_window - total_tokens)

if total_tokens <= context_window:
    print("Request fits within the context window")
else:
    print("Request exceeds the context window")

### 1.5 Temperature

In [ ]:
temperature_settings = [0, 0.2, 0.7, 1.0]

for temperature in temperature_settings:

    if temperature <= 0.2:
        behavior = "More focused and predictable"
    elif temperature <= 0.7:
        behavior = "Moderate variation"
    else:
        behavior = "More diverse and less predictable"

    print(f"Temperature: {temperature} | Behavior: {behavior}")

### 1.6 First LLM call - answering from a provided document (grounded prompt)

In [ ]:
policy_document = """
Healthcare Insurance Policy 2026

1. Claims must be submitted within 30 days
   from the date of hospital discharge.

2. The claim must include hospital bills,
   discharge summary and prescription details.

3. Incomplete claims may be rejected
   or returned for additional information.
"""

question = "What is the claim submission deadline?"

answer = chat(
    messages=[
        {
            "role": "system",
            "content": (
                "You are a healthcare insurance assistant. "
                "Answer only using the provided document. "
                "If information is missing, say it is not specified."
            ),
        },
        {"role": "user", "content": f"Document:{policy_document}\n\nQuestion:{question}"},
    ],
    temperature=0,
    max_tokens=250,
    timeout=60,
)

print(answer)

## Part 2 - Resume RAG pipeline

### 2.1 Create schema and volume
Upload your resume PDFs to the volume path printed below.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")

print("Catalog:", CATALOG)
print("Volume path:", VOLUME_PATH)

In [ ]:
display(dbutils.fs.ls(VOLUME_PATH))

### 2.2 Bronze - ingest raw PDFs
Uses `overwrite` so re-running the notebook does not duplicate documents.

In [ ]:
from pyspark.sql.functions import col, current_timestamp

bronze_df = (
    spark.read
    .format("binaryFile")
    .option("pathGlobFilter", "*.pdf")
    .load(VOLUME_PATH)
    .withColumn("source_file", col("path"))
    .withColumn("file_name", col("_metadata.file_name"))
    .withColumn("ingested", current_timestamp())
)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print("Bronze ingestion completed:", BRONZE_TABLE)

# Drop the binary column before displaying (keeps the notebook small)
display(spark.table(BRONZE_TABLE).drop("content"))

### 2.3 Silver - extract text from PDFs

In [ ]:
from io import BytesIO
from pypdf import PdfReader
from pyspark.sql.functions import col, udf, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType


def extract_pdf(pdf_bytes):
    """Return (text, page_count, status, error_message) for one PDF."""
    try:
        pdf = PdfReader(BytesIO(pdf_bytes))
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)
        return (text, len(pdf.pages), "SUCCESS", None)
    except Exception as e:
        return (None, 0, "FAILED", str(e)[:1000])


result_schema = StructType([
    StructField("extracted_text", StringType(), True),
    StructField("page_count", IntegerType(), True),
    StructField("extraction_status", StringType(), True),
    StructField("error_message", StringType(), True),
])

extract_pdf_udf = udf(extract_pdf, result_schema)

silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("extraction_result", extract_pdf_udf(col("content")))
    .select(
        col("path").alias("source_path"),
        col("modificationTime"),
        col("length").alias("file_size_bytes"),
        col("extraction_result.extracted_text"),
        col("extraction_result.page_count"),
        col("extraction_result.extraction_status"),
        col("extraction_result.error_message"),
        current_timestamp().alias("processed_at"),
    )
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver table created:", SILVER_TABLE)

#### Silver quality check

In [ ]:
from pyspark.sql.functions import count, sum as spark_sum, when, col

silver = spark.table(SILVER_TABLE)
silver.printSchema()

display(
    silver.groupBy("extraction_status").agg(
        count("*").alias("total_documents"),
        spark_sum(when(col("extracted_text").isNotNull(), 1).otherwise(0)).alias("documents_with_text"),
    )
)

### 2.4 Gold - chunk the text
- `chunk_overlap=150` so skills lists / job entries cut at a boundary still appear whole in one chunk.
- `chunk_id` is derived from `resume_id` + `chunk_index`, so it stays stable if the splitter settings change.

In [ ]:
from pyspark.sql.functions import col, udf, explode, sha2, concat_ws
from pyspark.sql.types import StringType, ArrayType, StructType, StructField, IntegerType
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
    length_function=len,
    keep_separator=True,
)

chunk_schema = ArrayType(
    StructType([
        StructField("chunk_text", StringType(), True),
        StructField("chunk_index", IntegerType(), True),
    ])
)


def create_chunks(text):
    if not text:
        return []
    return [
        {"chunk_text": chunk, "chunk_index": i}
        for i, chunk in enumerate(splitter.split_text(text))
    ]


create_chunks_udf = udf(create_chunks, chunk_schema)

gold_df = (
    spark.table(SILVER_TABLE)
    .where(col("extraction_status") == "SUCCESS")
    .withColumn("chunk", explode(create_chunks_udf(col("extracted_text"))))
    .select(
        col("source_path"),
        col("chunk.chunk_text").alias("chunk_text"),
        col("chunk.chunk_index").alias("chunk_index"),
    )
    .withColumn("resume_id", sha2(col("source_path"), 256))
    .withColumn(
        "chunk_id",
        sha2(concat_ws("||", col("resume_id"), col("chunk_index").cast("string")), 256),
    )
    .select("chunk_id", "resume_id", "source_path", "chunk_index", "chunk_text")
)

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print("Gold chunks table created:", GOLD_TABLE)
display(spark.table(GOLD_TABLE).sort("resume_id", "chunk_index").limit(15))

#### Chunk statistics per resume

In [ ]:
display(spark.sql(f"""
    SELECT
        resume_id,
        COUNT(*) AS total_chunks,
        MIN(LENGTH(chunk_text)) AS min_chunk_length,
        MAX(LENGTH(chunk_text)) AS max_chunk_length
    FROM {GOLD_TABLE}
    GROUP BY resume_id
"""))

### 2.5 Embeddings
First a quick connectivity test, then embed every chunk using the shared `embed_texts` helper.

In [ ]:
test_vector = embed_texts(["Azure Data Engineer with PySpark and SQL experience"])[0]

print("Dimensions:", len(test_vector))
print(test_vector[:10])

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType

# Collected to the driver - fine for a small learning project, not for large corpora
chunk_records = (
    spark.table(GOLD_TABLE)
    .select("chunk_id", "chunk_text")
    .where(F.col("chunk_text").isNotNull() & (F.length("chunk_text") > 0))
    .collect()
)

vectors = embed_texts([r["chunk_text"] for r in chunk_records], verbose=True)

embedded_records = [(r["chunk_id"], vec) for r, vec in zip(chunk_records, vectors)]

embedding_schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False),
])

embeddings_df = spark.createDataFrame(embedded_records, schema=embedding_schema)
display(embeddings_df.limit(5))

#### Join embeddings to chunks and save

In [ ]:
embedded_resume_df = (
    spark.table(GOLD_TABLE).join(embeddings_df, on="chunk_id", how="inner")
)

(
    embedded_resume_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(EMBEDDED_TABLE)
)

spark.sql(f"ALTER TABLE {EMBEDDED_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("Saved:", EMBEDDED_TABLE)

#### Validation

In [ ]:
display(spark.sql(f"""
    SELECT
        COUNT(*) AS total_chunks,
        COUNT(embedding) AS embedded_chunks,
        COUNT(DISTINCT chunk_id) AS unique_chunk_ids,
        MIN(size(embedding)) AS min_dimensions,
        MAX(size(embedding)) AS max_dimensions
    FROM {EMBEDDED_TABLE}
"""))

## Part 3 - Retrieval and answer generation

### 3.1 In-memory vector search
Cosine similarity over the stored embeddings with NumPy. This avoids overwriting internal
attributes of LangChain's `InMemoryVectorStore`, which can break between versions.
For larger corpora, move to **Databricks Vector Search**.

In [ ]:
from langchain_core.documents import Document

rows = spark.table(EMBEDDED_TABLE).collect()

docs = [
    Document(
        page_content=row["chunk_text"],
        metadata={
            "source": row["source_path"],
            "resume_id": row["resume_id"],
            "chunk_id": row["chunk_id"],
            "chunk_index": row["chunk_index"],
        },
    )
    for row in rows
]

matrix = np.array([row["embedding"] for row in rows], dtype=np.float32)
matrix = matrix / np.clip(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12, None)


def similarity_search(query, k=20):
    """Return the top-k (Document, cosine_score) pairs for a query."""
    q = np.array(embed_texts([query])[0], dtype=np.float32)
    q = q / max(np.linalg.norm(q), 1e-12)
    scores = matrix @ q
    top = np.argsort(-scores)[:k]
    return [(docs[i], float(scores[i])) for i in top]


print("Chunks in index:", len(docs))

### 3.2 Retrieve
`k=20` (up from 10): skill-style questions span many resumes, and a small `k` can miss candidates.

In [ ]:
query = "Find candidates with Azure Databricks and PySpark experience"

results = similarity_search(query, k=20)

for doc, score in results:
    print(f"{score:.3f} | {doc.metadata['resume_id'][:8]} | chunk {doc.metadata['chunk_index']}")

### 3.3 Build the context (with a size budget)

In [ ]:
MAX_CONTEXT_CHARS = 12000  # rough guard so the prompt stays inside the model's context window

context = ""
used_chunks = 0

for i, (doc, score) in enumerate(results, start=1):
    block = f"""
Resume Chunk {i}
Resume ID: {doc.metadata["resume_id"]}
Source: {doc.metadata["source"]}

Content:
{doc.page_content}

---
"""
    if len(context) + len(block) > MAX_CONTEXT_CHARS:
        break
    context += block
    used_chunks += 1

print(f"Using {used_chunks} of {len(results)} retrieved chunks ({len(context)} characters)")

### 3.4 RAG prompt

In [ ]:
rag_prompt = f"""
You are a resume analysis assistant.

Answer the user's question using ONLY the resume content
provided below.

Rules:
- Do not invent candidate skills or experience.
- If evidence is insufficient, clearly say so.
- Identify the resume ID supporting each finding.
- Distinguish explicit experience from inferred skills.
- The retrieved content is only a sample of all resumes; do not claim that
  other candidates lack a skill just because it is not shown here.

Retrieved Resume Content:
{context}

User Question:
{query}
"""

### 3.5 Generate the answer

In [ ]:
answer = chat(messages=[{"role": "user", "content": rag_prompt}], temperature=0)
print(answer)